# 🤖 ARIMA Training v3 — Multi-country Electricity Forecasting

**Mục tiêu:** Train mô hình **ARIMA/SARIMAX** trên bộ dữ liệu đa quốc gia  
và lưu checkpoint để làm **pretrained model** cho bước Transfer Learning sang Việt Nam.

---
| Mục | Thông tin |
|---|---|
| Dataset | `tft_premodel_dataset_EDA.csv` (34,614 rows × 30 cols) |
| Entities | 20 quốc gia |
| Series target | Coal, Gas, Hydro, Solar, Wind, Bioenergy, Nuclear, Other Fossil, Other Renewables |
| Training approach | Grid search (p,d,q) per series |
| Model output | Pickle models + config JSON |
| Checkpoint output | `checkpoint/arima_v3_models_*.pkl`, `checkpoint/arima_v3_config.json` |

In [1]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — IMPORTS & CONFIGURATION
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings, os, json, pickle
from pathlib import Path
from tqdm import tqdm
warnings.filterwarnings('ignore')

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, acf, pacf, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print(f'✅ Pandas      : {pd.__version__}')
print(f'   Numpy      : {np.__version__}')
print(f'   Statsmodels: statsmodels imported')
print(f'   Sklearn    : sklearn imported')

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR  = Path(r'C:\Users\ADMIN\OneDrive - Hanoi University of Mining and Geology\Documents\NCKH\TFT-GreenPower-Forecasting')
DATA_PATH = BASE_DIR / 'data' / 'processed' / 'training_data' / 'tft_premodel_dataset_EDA.csv'
CKPT_DIR  = BASE_DIR / 'checkpoint'
LOG_DIR   = BASE_DIR / 'arima_logs' / 'arima_v3'
CKPT_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f'   Data      : {DATA_PATH}')
print(f'   Checkpoint: {CKPT_DIR}')

# ── Random Seed ────────────────────────────────────────────────────────────────
np.random.seed(42)
print(f'✅ Random seed set to 42')

✅ Pandas      : 2.3.3
   Numpy      : 1.26.4
   Statsmodels: statsmodels imported
   Sklearn    : sklearn imported
   Data      : C:\Users\ADMIN\OneDrive - Hanoi University of Mining and Geology\Documents\NCKH\TFT-GreenPower-Forecasting\data\processed\training_data\tft_premodel_dataset_EDA.csv
   Checkpoint: C:\Users\ADMIN\OneDrive - Hanoi University of Mining and Geology\Documents\NCKH\TFT-GreenPower-Forecasting\checkpoint
✅ Random seed set to 42


In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — HYPERPARAMETERS & CONFIG
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CFG = dict(
    # ── Dataset ───────────────────────────────────────────────────────────────
    val_cutoff_months     = 12,   # dùng 12 tháng cuối làm validation
    min_series_length     = 30,   # tối thiểu độ dài chuỗi

    # ── ARIMA order search ────────────────────────────────────────────────────
    p_range              = (0, 3),   # AR order
    d_range              = (0, 2),   # differencing
    q_range              = (0, 3),   # MA order
    
    # ── Seasonal ARIMA (optional) ─────────────────────────────────────────────
    use_seasonal         = False,    # set True nếu muốn SARIMAX (12 tháng)
    seasonal_period      = 12,

    # ── Training ──────────────────────────────────────────────────────────────
    max_grid_search_time = 60,       # timeout (giây) cho grid search
    seed                 = 42,
)

print('📋 Cấu hình ARIMA v3:')
for k, v in CFG.items():
    print(f'   {k:<28}: {v}')

📋 Cấu hình ARIMA v3:
   val_cutoff_months           : 12
   min_series_length           : 30
   p_range                     : (0, 3)
   d_range                     : (0, 2)
   q_range                     : (0, 3)
   use_seasonal                : False
   seasonal_period             : 12
   max_grid_search_time        : 60
   seed                        : 42


In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — LOAD & PREPROCESS DATA
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
df_raw = pd.read_csv(DATA_PATH)
df_raw['date'] = pd.to_datetime(df_raw['date'])
print(f'Raw shape: {df_raw.shape}')

# ── Chỉ giữ các series là nguồn điện thực sự ──────────────────────────────────
TARGET_SERIES = [
    'Coal', 'Gas', 'Hydro', 'Solar', 'Wind',
    'Bioenergy', 'Nuclear', 'Other Fossil', 'Other Renewables',
]
df = df_raw[df_raw['series'].isin(TARGET_SERIES)].copy()
print(f'✅ Filtered to target series: {len(df)} rows')

# ── Fill NaN từ lag features ──────────────────────────────────────────────────
# ARIMA dùng target chính, nên chỉ cần chuỗi generation_TWh sạch
df = df.sort_values(['entity', 'series', 'date'])
df['time_idx'] = df.groupby(['entity','series'])['date'].rank(method='dense').astype(int) - 1

# ── Lọc series đủ độ dài ──────────────────────────────────────────────────────
counts  = df.groupby(['entity','series']).size()
valid   = counts[counts >= CFG['min_series_length']].reset_index()[['entity','series']]
df      = df.merge(valid, on=['entity','series'])

# ── Kiểm tra NaN ──────────────────────────────────────────────────────────────
nan_count = df['generation_TWh'].isnull().sum()
print(f'✅ Target series (generation_TWh) NaN: {nan_count}')
print(f'   Shape: {df.shape} | Groups: {df.groupby(["entity","series"]).ngroups}')
print(f'   Date range: {df["date"].min().date()} → {df["date"].max().date()}')

Raw shape: (30290, 30)
✅ Filtered to target series: 13878 rows
✅ Target series (generation_TWh) NaN: 0
   Shape: (13859, 30) | Groups: 151
   Date range: 2018-01-01 → 2025-12-01


In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — TRAIN/VALIDATION SPLIT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
training_cutoff = int(df['time_idx'].max()) - CFG['val_cutoff_months']
val_cutoff_date = df['date'].max() - pd.DateOffset(months=CFG['val_cutoff_months'])

df_train = df[df['time_idx'] <= training_cutoff].copy()
df_val   = df[df['time_idx'] > training_cutoff].copy()

print(f'📅 Train/Val Split:')
print(f'   training_cutoff (time_idx): {training_cutoff}')
print(f'   val_cutoff_date           : {val_cutoff_date.date()}')
print(f'   Train rows                : {len(df_train):,}')
print(f'   Val rows                  : {len(df_val):,}')

📅 Train/Val Split:
   training_cutoff (time_idx): 83
   val_cutoff_date           : 2024-12-01
   Train rows                : 12,451
   Val rows                  : 1,408


In [5]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5 — STATIONARITY CHECK & DIFFERENCING
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def check_stationarity(ts, name, verbose=False):
    """Check stationarity using ADF test."""
    try:
        result = adfuller(ts.dropna(), autolag='AIC')
        if verbose:
            print(f'{name}: ADF stat={result[0]:.4f}, p-value={result[1]:.4f}')
        return result[1] < 0.05  # True = stationary
    except:
        return False

# Test một vài series
print('🔍 Stationarity Check (ADF test, 5% level):')
sample_entities = df['entity'].unique()[:3]
sample_series = ['Coal', 'Wind', 'Solar']

for ent in sample_entities:
    for ser in sample_series:
        try:
            ts = df_train[(df_train['entity']==ent) & (df_train['series']==ser)]['generation_TWh']
            if len(ts) > 10:
                is_stat = check_stationarity(ts, f'{ent}-{ser}', verbose=True)
        except:
            pass

🔍 Stationarity Check (ADF test, 5% level):
Australia-Coal: ADF stat=-2.0523, p-value=0.2641
Australia-Wind: ADF stat=-1.7402, p-value=0.4105
Australia-Solar: ADF stat=0.3706, p-value=0.9804
Austria-Coal: ADF stat=-1.0003, p-value=0.7532
Austria-Wind: ADF stat=-5.9968, p-value=0.0000
Austria-Solar: ADF stat=5.0240, p-value=1.0000
Chile-Coal: ADF stat=-0.3590, p-value=0.9167
Chile-Wind: ADF stat=0.1915, p-value=0.9718
Chile-Solar: ADF stat=0.2528, p-value=0.9751


In [6]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 6 — GRID SEARCH FOR OPTIMAL (p,d,q) ORDER
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def find_best_arima_order(ts, p_range, d_range, q_range, max_time=60):
    """Grid search for best ARIMA order using AIC."""
    import time
    
    ts_clean = ts.dropna()
    if len(ts_clean) < 15:
        return None
    
    best_aic = np.inf
    best_order = None
    start_time = time.time()
    
    for p in range(p_range[0], p_range[1] + 1):
        for d in range(d_range[0], d_range[1] + 1):
            for q in range(q_range[0], q_range[1] + 1):
                if time.time() - start_time > max_time:
                    return best_order
                
                try:
                    model = ARIMA(ts_clean, order=(p, d, q))
                    result = model.fit()
                    if result.aic < best_aic:
                        best_aic = result.aic
                        best_order = (p, d, q)
                except:
                    pass
    
    return best_order

print('⏳ Starting grid search for optimal orders...')
print('This may take a few minutes depending on data size.')

⏳ Starting grid search for optimal orders...
This may take a few minutes depending on data size.


In [7]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 7 — TRAIN ARIMA MODELS PER SERIES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
models_dict = {}  # {(entity, series): fitted_model}
order_dict = {}   # {(entity, series): (p, d, q)}
metrics_dict = {} # {(entity, series): {mae, rmse, smape, wape}}

groups = df_train.groupby(['entity', 'series'])
print(f'Training {len(groups)} ARIMA models...')

for (entity, series), group_df in tqdm(groups, desc='ARIMA Training'):
    ts_train = group_df['generation_TWh'].values
    
    if len(ts_train) < 15:
        continue
    
    # Find best order
    best_order = find_best_arima_order(
        pd.Series(ts_train),
        CFG['p_range'], CFG['d_range'], CFG['q_range'],
        max_time=CFG['max_grid_search_time']
    )
    
    if best_order is None:
        best_order = (1, 1, 1)  # default fallback
    
    order_dict[(entity, series)] = best_order
    
    # Fit model on full training set
    try:
        model = ARIMA(ts_train, order=best_order)
        fitted = model.fit()
        models_dict[(entity, series)] = fitted
    except Exception as e:
        print(f'Failed to fit {entity}-{series}: {e}')

print(f'✅ Fitted {len(models_dict)} models')

Training 151 ARIMA models...


ARIMA Training:   0%|          | 0/151 [00:00<?, ?it/s]c:\Users\ADMIN\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\ADMIN\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\ADMIN\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\ADMIN\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Chec

✅ Fitted 151 models


In [9]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 8 — VALIDATION FORECAST & METRICS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
all_actuals = []
all_preds = []

for (entity, series), fitted_model in tqdm(models_dict.items(), desc='Validation Forecast'):
    # Get validation data
    val_data = df_val[(df_val['entity']==entity) & (df_val['series']==series)].sort_values('date')
    
    if len(val_data) == 0:
        continue
    
    actuals = val_data['generation_TWh'].values
    
    # Forecast
    try:
        preds = fitted_model.forecast(steps=len(actuals))
        all_actuals.extend(actuals)
        all_preds.extend(preds)
        
        # Calculate metrics for this series
        mae = np.mean(np.abs(actuals - preds))
        rmse = np.sqrt(np.mean((actuals - preds)**2))
        smape = np.mean(2 * np.abs(actuals - preds) / (np.abs(actuals) + np.abs(preds) + 1e-8)) * 100
        wape = np.sum(np.abs(actuals - preds)) / (np.sum(np.abs(actuals)) + 1e-8) * 100
        r2 = r2_score(actuals, preds)
        
        metrics_dict[(entity, series)] = {
            'mae': mae, 'rmse': rmse, 'smape': smape, 'wape': wape, 'r2': r2
        }
    except Exception as e:
        print(f'Failed to forecast {entity}-{series}: {e}')

# Overall metrics
if all_actuals:
    y_true = np.array(all_actuals)
    y_pred = np.array(all_preds)
    overall_mae = np.mean(np.abs(y_true - y_pred))
    overall_rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    overall_smape = np.mean(2 * np.abs(y_true - y_pred) / 
                            (np.abs(y_true) + np.abs(y_pred) + 1e-8)) * 100
    overall_wape = np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + 1e-8) * 100
    overall_r2 = r2_score(y_true, y_pred)

    print(f'\n📊 Overall Validation Metrics:')
    print(f'   MAE   : {overall_mae:.4f} TWh')
    print(f'   RMSE  : {overall_rmse:.4f} TWh')
    print(f'   SMAPE : {overall_smape:.2f}%')
    print(f'   WAPE  : {overall_wape:.2f}%')
    print(f'   R²    : {overall_r2:.4f}')

Validation Forecast: 100%|██████████| 151/151 [00:00<00:00, 455.94it/s]


📊 Overall Validation Metrics:
   MAE   : 0.3415 TWh
   RMSE  : 0.6935 TWh
   SMAPE : 29.34%
   WAPE  : 14.86%
   R²    : 0.9695


In [10]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 9 — SAVE MODELS & CONFIG JSON
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Save models
models_pkl = CKPT_DIR / 'arima_v3_models.pkl'
with open(models_pkl, 'wb') as f:
    pickle.dump(models_dict, f)
print(f'✅ Models saved → {models_pkl}')

# Save order dict
order_pkl = CKPT_DIR / 'arima_v3_orders.pkl'
with open(order_pkl, 'wb') as f:
    pickle.dump(order_dict, f)
print(f'✅ Orders saved → {order_pkl}')

# Save config JSON
model_config = {
    'version'                   : 'arima_v3',
    'models_pkl'                : str(models_pkl),
    'orders_pkl'                : str(order_pkl),
    'target'                    : 'generation_TWh',
    'group_ids'                 : ['entity', 'series'],
    'target_series'             : TARGET_SERIES,
    'p_range'                   : CFG['p_range'],
    'd_range'                   : CFG['d_range'],
    'q_range'                   : CFG['q_range'],
    'use_seasonal'              : CFG['use_seasonal'],
    'val_metrics'               : {
        'mae': float(overall_mae) if 'overall_mae' in locals() else None,
        'rmse': float(overall_rmse) if 'overall_rmse' in locals() else None,
        'smape': float(overall_smape) if 'overall_smape' in locals() else None,
        'wape': float(overall_wape) if 'overall_wape' in locals() else None,
        'r2': float(overall_r2) if 'overall_r2' in locals() else None,
    },
    'train_entities'            : sorted(df['entity'].unique().tolist()),
    'train_series'              : sorted(df['series'].unique().tolist()),
    'train_date_range'          : [str(df['date'].min().date()), str(df['date'].max().date())],
    'num_models'                : len(models_dict),
}

config_path = CKPT_DIR / 'arima_v3_config.json'
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(model_config, f, indent=2, ensure_ascii=False)

print(f'✅ Config saved → {config_path}')

✅ Models saved → C:\Users\ADMIN\OneDrive - Hanoi University of Mining and Geology\Documents\NCKH\TFT-GreenPower-Forecasting\checkpoint\arima_v3_models.pkl
✅ Orders saved → C:\Users\ADMIN\OneDrive - Hanoi University of Mining and Geology\Documents\NCKH\TFT-GreenPower-Forecasting\checkpoint\arima_v3_orders.pkl
✅ Config saved → C:\Users\ADMIN\OneDrive - Hanoi University of Mining and Geology\Documents\NCKH\TFT-GreenPower-Forecasting\checkpoint\arima_v3_config.json


In [11]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 10 — PRINT ORDER SUMMARY TABLE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
order_df = pd.DataFrame([
    {'entity': k[0], 'series': k[1], 'p': v[0], 'd': v[1], 'q': v[2]}
    for k, v in order_dict.items()
])

print('\n📋 Top 15 Orders by (p,d,q):')
print(order_df.head(15).to_string(index=False))
print(f'\nTotal unique (p,d,q) combinations: {len(order_df)}')


📋 Top 15 Orders by (p,d,q):
   entity           series  p  d  q
Australia        Bioenergy  3  0  0
Australia             Coal  3  1  2
Australia              Gas  0  0  3
Australia            Hydro  3  0  0
Australia     Other Fossil  2  0  3
Australia            Solar  2  1  3
Australia             Wind  0  1  3
  Austria        Bioenergy  2  0  1
  Austria             Coal  2  0  3
  Austria              Gas  3  0  2
  Austria            Hydro  2  0  3
  Austria     Other Fossil  1  0  2
  Austria Other Renewables  0  1  0
  Austria            Solar  3  1  3
  Austria             Wind  2  1  3

Total unique (p,d,q) combinations: 151


In [12]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 11 — TRAINING SUMMARY & NEXT STEPS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('╔══════════════════════════════════════════════════════════════╗')
print('║                   KẾT QUẢ TRAINING ARIMA                     ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  Models trained    : {len(models_dict):<40} ║')
print(f'║  MAE              : {overall_mae:.4f} TWh{" "*35} ║')
print(f'║  RMSE             : {overall_rmse:.4f} TWh{" "*34} ║')
print(f'║  SMAPE            : {overall_smape:.2f}%{" "*38} ║')
print(f'║  WAPE             : {overall_wape:.2f}%{" "*37} ║')
print(f'║  R²               : {overall_r2:.4f}{" "*35} ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  Models checkpoint: arima_v3_models.pkl                     ║')
print(f'║  Orders checkpoint: arima_v3_orders.pkl                     ║')
print(f'║  Config JSON      : arima_v3_config.json                    ║')
print('╠══════════════════════════════════════════════════════════════╣')
print('║  NEXT STEPS — Transfer Learning sang VN:                     ║')
print('║  1. Load: models_dict từ arima_v3_models.pkl                 ║')
print('║  2. Load: order_dict từ arima_v3_orders.pkl                  ║')
print('║  3. Fine-tune trên dữ liệu Việt Nam trong notebook transfer  ║')
print('╚══════════════════════════════════════════════════════════════╝')

╔══════════════════════════════════════════════════════════════╗
║                   KẾT QUẢ TRAINING ARIMA                     ║
╠══════════════════════════════════════════════════════════════╣
║  Models trained    : 151                                      ║
║  MAE              : 0.3415 TWh                                    ║
║  RMSE             : 0.6935 TWh                                   ║
║  SMAPE            : 29.34%                                       ║
║  WAPE             : 14.86%                                      ║
║  R²               : 0.9695                                    ║
╠══════════════════════════════════════════════════════════════╣
║  Models checkpoint: arima_v3_models.pkl                     ║
║  Orders checkpoint: arima_v3_orders.pkl                     ║
║  Config JSON      : arima_v3_config.json                    ║
╠══════════════════════════════════════════════════════════════╣
║  NEXT STEPS — Transfer Learning sang VN:                     ║
║  1. Load